In [13]:
import pandas as pd
import networkx as nx
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import ndlib.models.ModelConfig as mc
import ndlib.models.epidemics as ep
from community import community_louvain
from datetime import timedelta
import matplotlib.pyplot as plt

## 1. Tiền xử lý dữ liệu

In [2]:
# Đọc từng file từ thư mục MXH_Dataset
train_df = pd.read_csv("../Dataset/train.csv")
segment_status_df = pd.read_csv("../Dataset/segment_status.csv")


In [3]:
# Kiểm tra số dòng, số cột của từng file
for name, df in [("segment_status", segment_status_df), 
                 ("train", train_df)]:
    print(f"{name}: {df.info}")

segment_status: <bound method DataFrame.info of          _id                updated_at  segment_id  velocity
0          0  2020-07-03T14:55:31.869Z       24845        20
1          1  2020-07-03T15:02:56.048Z       33923        10
2          2  2020-07-04T08:15:52.696Z       33824         5
3          3  2020-07-04T08:15:59.903Z       33824         5
4          4  2020-07-04T08:16:08.201Z       33824         5
...      ...                       ...         ...       ...
90933  90933  2021-04-22T06:52:39.280Z       52247         1
90934  90934  2021-04-22T06:52:52.501Z       52247         1
90935  90935  2021-04-22T06:53:02.335Z       52247         1
90936  90936  2021-04-22T06:53:14.294Z       52247         1
90937  90937  2021-04-22T06:53:27.300Z       52247         1

[90938 rows x 4 columns]>
train: <bound method DataFrame.info of          _id  segment_id        date  weekday        period LOS   s_node_id  \
0          0          26  2021-04-16        4   period_0_30   A   366428456

In [4]:
# Chuyển đổi thời gian
train_df['date'] = pd.to_datetime(train_df['date'])
segment_status_df['updated_at'] = pd.to_datetime(segment_status_df['updated_at']).dt.tz_localize(None)

In [6]:
# Sort trước khi merge
train_df = train_df.sort_values(by='date')
segment_status_df = segment_status_df.sort_values(by='updated_at')

# Merge gần đúng theo thời gian và segment_id
merged_df = pd.merge_asof(
    train_df,
    segment_status_df,
    by='segment_id',
    left_on='date',
    right_on='updated_at',
    direction='nearest',
    tolerance=pd.Timedelta(minutes=30)
)


In [9]:
# Tạo đồ thị tổng hợp
G = nx.Graph()
for _, row in merged_df.iterrows():
    G.add_node(row['s_node_id'], long=row['long_snode'], lat=row['lat_snode'])
    G.add_node(row['e_node_id'], long=row['long_enode'], lat=row['lat_enode'])
    G.add_edge(
        row['s_node_id'], row['e_node_id'],
        segment_id=row['segment_id'],
        length=row['length'],
        velocity=row['velocity'] if pd.notnull(row['velocity']) else 0,
        LOS=row['LOS'],
        street_type=row['street_type'],
        street_level=row['street_level']
    )

# Kết quả transform
num_nodes = G.number_of_nodes()
num_edges = G.number_of_edges()
avg_degree = np.mean([d for _, d in G.degree()])
print(f"Số node (giao lộ): {num_nodes}")
print(f"Số cạnh (đoạn đường): {num_edges}")
print(f"Độ trung bình: {avg_degree:.2f}")


Số node (giao lộ): 11314
Số cạnh (đoạn đường): 8753
Độ trung bình: 1.55


In [10]:
# 1. Betweenness Centrality
betweenness_centrality = nx.betweenness_centrality(G)
avg_betweenness = np.mean(list(betweenness_centrality.values()))
top_betweenness = sorted(betweenness_centrality.items(), key=lambda x: x[1], reverse=True)[:5]

# 2. Closeness Centrality
closeness_centrality = nx.closeness_centrality(G)
avg_closeness = np.mean(list(closeness_centrality.values()))
top_closeness = sorted(closeness_centrality.items(), key=lambda x: x[1], reverse=True)[:5]
center_node = max(closeness_centrality.items(), key=lambda x: x[1])[0]


In [14]:
# 3. Lan truyền tắc nghẽn
los_map = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4, 'F': 5}
model = ep.SIRModel(G)
config = mc.Configuration()
config.add_model_parameter('beta', 0.01)
config.add_model_parameter('gamma', 0.05)
for node in G.nodes():
    edges = G.edges(node, data=True)
    if any(data['LOS'] in ['E', 'F'] for _, _, data in edges):
        config.add_node_configuration('status', node, 'Infected')
    else:
        config.add_node_configuration('status', node, 'Susceptible')
model.set_initial_status(config)
iterations = model.iteration_bunch(100)
congested_nodes = [it['node_count'][1] for it in iterations]
max_congested = max(congested_nodes)

C:\Users\ASUS\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\ndlib\models\DiffusionModel.py:120: UserWarning: Initial infection missing: a random sample of 5% of graph nodes will be set as infected
  warnings.warn('Initial infection missing: a random sample of 5% of graph nodes will be set as infected')


In [15]:
# 4. Độ tương đồng
# Node similarity
for node in G.nodes():
    edges = G.edges(node, data=True)
    degrees = len(edges)
    avg_los = np.mean([los_map[data['LOS']] for _, _, data in edges]) if edges else 0
    G.nodes[node]['degree'] = degrees
    G.nodes[node]['avg_los'] = avg_los
node_features = np.array([[G.nodes[n]['degree'], G.nodes[n]['avg_los']] for n in G.nodes()])
node_similarity = cosine_similarity(node_features)
for node in G.nodes():
    neighbors = list(G.neighbors(node))
    if neighbors:
        G.nodes[node]['similarity'] = np.mean([node_similarity[list(G.nodes).index(node), list(G.nodes).index(n)] for n in neighbors])
    else:
        G.nodes[node]['similarity'] = 0
# Edge similarity
for u, v, data in G.edges(data=True):
    G.edges[u, v]['velocity'] = data['velocity']
    G.edges[u, v]['los_encoded'] = los_map[data['LOS']]
edge_features = np.array([[G.edges[u, v]['velocity'], G.edges[u, v]['los_encoded']] for u, v in G.edges()])
edge_similarity = cosine_similarity(edge_features)
for i, (u, v) in enumerate(G.edges()):
    neighbor_edges = [(u1, v1) for u1, v1 in G.edges() if u1 == u or v1 == u or u1 == v or v1 == v]
    if neighbor_edges:
        neighbor_indices = [list(G.edges()).index(e) for e in neighbor_edges if e != (u, v)]
        G.edges[u, v]['spatial_similarity'] = np.mean([edge_similarity[i, j] for j in neighbor_indices]) if neighbor_indices else 0
    else:
        G.edges[u, v]['spatial_similarity'] = 0
avg_node_similarity = np.mean([G.nodes[n]['similarity'] for n in G.nodes()])
avg_edge_similarity = np.mean([G.edges[u, v]['spatial_similarity'] for u, v in G.edges()])

In [16]:
# 5. Cộng đồng và Modularity
partition = community_louvain.best_partition(G)
modularity = community_louvain.modularity(partition, G)
num_communities = len(set(partition.values()))

In [ ]:
# Vẽ biểu đồ
plt.hist([G.nodes[n]['similarity'] for n in G.nodes()], bins=20)
plt.title("Phân bố độ tương đồng giao lộ")
plt.savefig("node_similarity_histogram.png")
plt.close()

plt.hist([G.edges[u, v]['spatial_similarity'] for u, v in G.edges()], bins=20)
plt.title("Phân bố độ tương đồng đoạn đường")
plt.savefig("edge_similarity_histogram.png")
plt.close()

plt.plot(congested_nodes)
plt.title("Lan truyền tắc nghẽn giao thông")
plt.xlabel("Thời gian")
plt.ylabel("Số giao lộ tắc nghẽn")
plt.savefig("congestion_spread.png")
plt.close()

In [19]:
# In kết quả
print(f"Betweenness centrality trung bình: {avg_betweenness:.4f}")
print(f"Top 5 giao lộ quan trọng: {top_betweenness}")
print(f"Closeness centrality trung bình: {avg_closeness:.4f}")
print(f"Top 5 giao lộ gần trung tâm: {top_closeness}")
print(f"Giao lộ trung tâm: {center_node}, Closeness: {closeness_centrality[center_node]:.4f}")
print(f"Số giao lộ tắc nghẽn tối đa: {max_congested}")
print(f"Độ tương đồng giao lộ trung bình: {avg_node_similarity:.4f}")
print(f"Độ tương đồng đoạn đường trung bình: {avg_edge_similarity:.4f}")
print(f"Số cộng đồng: {num_communities}")
print(f"Modularity: {modularity:.4f}")

Betweenness centrality trung bình: 0.0002
Top 5 giao lộ quan trọng: [(4632009841, 0.01676977944720916), (4628048104, 0.012493862782330957), (5788625780, 0.011441107123832252), (366383937, 0.010811648030293981), (2433653217, 0.010508335948949041)]
Closeness centrality trung bình: 0.0009
Top 5 giao lộ gần trung tâm: [(4632009841, 0.0053322933716990675), (4628048104, 0.005300390862029619), (2433653217, 0.005286205325445927), (366457975, 0.00527376855633753), (2302073743, 0.005251700895955117)]
Giao lộ trung tâm: 4632009841, Closeness: 0.0053
Số giao lộ tắc nghẽn tối đa: 565
Độ tương đồng giao lộ trung bình: 0.9688
Độ tương đồng đoạn đường trung bình: 0.4628
Số cộng đồng: 2725
Modularity: 0.9882
